In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")

In [ ]:
file_path = "data/retail-orders-raw.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

In [ ]:
print("Dataset Shape:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

In [ ]:
print("Missing Values:")
print(df.isnull().sum())

In [ ]:
print("Duplicate Rows:", df.duplicated().sum())

In [ ]:
# Convert date columns where available
date_columns = ["Order Date", "Ship Date"]

for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# Convert numerical columns
numeric_columns = ["Sales", "Quantity", "Discount", "Profit"]

for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("Data preparation completed.")

In [ ]:
numeric_df = df.select_dtypes(include=np.number)

descriptive_stats = numeric_df.describe().T

descriptive_stats["median"] = numeric_df.median()

descriptive_stats = descriptive_stats[
    ["count", "mean", "median", "std", "min", "25%", "50%", "75%", "max"]
]

descriptive_stats

## Descriptive Statistics

The descriptive statistics provide an overall understanding of the numerical
variables in the retail dataset.

Mean and median help identify the central tendency, while standard deviation
shows the spread of observations. Quartiles provide additional information
about how the values are distributed across the dataset.

Large differences between mean and median may indicate skewed distributions
or the presence of extreme observations.

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(df["Sales"].dropna(), bins=40, kde=True)

plt.title("Sales Distribution")
plt.xlabel("Sales")
plt.ylabel("Frequency")
plt.show()

In [ ]:
## Sales Distribution Analysis

The sales distribution is examined to understand the frequency of low,
medium, and high-value transactions. The shape of the distribution also
helps identify whether sales values are concentrated around a specific
range or affected by extreme observations.

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(df["Profit"].dropna(), bins=40, kde=True)

plt.title("Profit Distribution")
plt.xlabel("Profit")
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(df["Discount"].dropna(), bins=20, kde=True)

plt.title("Discount Distribution")
plt.xlabel("Discount")
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.histplot(df["Quantity"].dropna(), bins=20, kde=True)

plt.title("Quantity Distribution")
plt.xlabel("Quantity")
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))

sns.boxplot(x=df["Sales"])

plt.title("Sales Box Plot")
plt.xlabel("Sales")
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))

sns.boxplot(x=df["Profit"])

plt.title("Profit Box Plot")
plt.xlabel("Profit")
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))

sns.boxplot(x=df["Discount"])

plt.title("Discount Box Plot")
plt.xlabel("Discount")
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))

sns.boxplot(x=df["Quantity"])

plt.title("Quantity Box Plot")
plt.xlabel("Quantity")
plt.show()

## Outlier Analysis

Box plots were used to identify unusually high or low observations in
Sales, Profit, Discount, and Quantity.

These observations are not automatically removed because extreme values
may represent genuine business transactions. Instead, they are considered
during interpretation of the statistical results.

In [ ]:
correlation_matrix = numeric_df.corr()

correlation_matrix

In [ ]:
plt.figure(figsize=(10, 7))

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=0.5
)

plt.title("Correlation Heatmap")
plt.show()

## Correlation Analysis

The correlation matrix measures the strength and direction of linear
relationships between numerical variables.

Positive values indicate that two variables tend to increase together,
while negative values indicate an inverse relationship. Correlation does
not establish causation and should therefore be interpreted together with
the business context and other statistical tests.

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x="Sales",
    y="Profit",
    alpha=0.5
)

plt.title("Sales vs Profit")
plt.xlabel("Sales")
plt.ylabel("Profit")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x="Discount",
    y="Profit",
    alpha=0.5
)

plt.title("Discount vs Profit")
plt.xlabel("Discount")
plt.ylabel("Profit")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x="Quantity",
    y="Sales",
    alpha=0.5
)

plt.title("Quantity vs Sales")
plt.xlabel("Quantity")
plt.ylabel("Sales")
plt.show()

In [ ]:
category_summary = df.groupby("Category")[["Sales", "Profit"]].agg(
    ["sum", "mean"]
)

category_summary

In [ ]:
category_totals = df.groupby("Category")[["Sales", "Profit"]].sum()

category_totals.plot(
    kind="bar",
    figsize=(10, 6)
)

plt.title("Category-wise Sales and Profit")
plt.xlabel("Category")
plt.ylabel("Amount")
plt.xticks(rotation=0)
plt.legend(["Sales", "Profit"])
plt.show()

In [ ]:
region_summary = df.groupby("Region")[["Sales", "Profit"]].sum()

region_summary

In [ ]:
region_summary.plot(
    kind="bar",
    figsize=(10, 6)
)

plt.title("Region-wise Sales and Profit")
plt.xlabel("Region")
plt.ylabel("Amount")
plt.xticks(rotation=0)
plt.show()

In [ ]:
discount_profit = df[["Discount", "Profit"]].dropna()

r1, p1 = stats.pearsonr(
    discount_profit["Discount"],
    discount_profit["Profit"]
)

print("Pearson Correlation:", round(r1, 4))
print("P-value:", round(p1, 6))

if p1 < 0.05:
    print("Result: Reject H0 — statistically significant relationship detected.")
else:
    print("Result: Fail to reject H0 — statistically significant relationship not detected.")

### Hypothesis 1 Interpretation

Pearson correlation and its p-value are used to examine the relationship
between discount and profit.

A p-value below 0.05 indicates statistical evidence of a linear association
between the two variables. The direction of the correlation coefficient
indicates whether the relationship is positive or negative.

This analysis identifies association and should not be interpreted as proof
that discount directly causes changes in profit.

In [ ]:
quantity_sales = df[["Quantity", "Sales"]].dropna()

r2, p2 = stats.pearsonr(
    quantity_sales["Quantity"],
    quantity_sales["Sales"]
)

print("Pearson Correlation:", round(r2, 4))
print("P-value:", round(p2, 6))

if p2 < 0.05:
    print("Result: Reject H0 — statistically significant relationship detected.")
else:
    print("Result: Fail to reject H0 — statistically significant relationship not detected.")

### Hypothesis 2 Interpretation

The Pearson correlation test evaluates whether Quantity and Sales have a
statistically significant linear relationship.

The correlation coefficient indicates the direction and strength of the
relationship, while the p-value determines whether the observed association
is statistically significant at the 5% significance level.

In [ ]:
category_profit = df[
    ["Category", "Profit"]
].dropna()

groups = [
    group["Profit"].values
    for _, group in category_profit.groupby("Category")
]

anova_result = stats.f_oneway(*groups)

print("ANOVA F-statistic:", round(anova_result.statistic, 4))
print("P-value:", round(anova_result.pvalue, 6))

if anova_result.pvalue < 0.05:
    print("Result: Reject H0 — category-level mean profit differs significantly.")
else:
    print("Result: Fail to reject H0 — significant category-level difference was not detected.")

In [ ]:
category_profit_avg = (
    df.groupby("Category")["Profit"]
    .mean()
    .sort_values(ascending=False)
)

category_profit_avg

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    x=category_profit_avg.index,
    y=category_profit_avg.values
)

plt.title("Average Profit by Category")
plt.xlabel("Category")
plt.ylabel("Average Profit")
plt.show()

### Hypothesis 3 Interpretation

A one-way ANOVA test was applied to compare the average profit across
different product categories.

If the p-value is below 0.05, the result provides statistical evidence
that at least one category has a different mean profit. ANOVA alone does
not identify which specific category pairs differ; a post-hoc test would
be required for that purpose.

In [ ]:
hypothesis_summary = pd.DataFrame({
    "Hypothesis": [
        "Discount vs Profit",
        "Quantity vs Sales",
        "Category Profit Differences"
    ],
    "Test": [
        "Pearson Correlation",
        "Pearson Correlation",
        "One-Way ANOVA"
    ],
    "Statistic": [
        r1,
        r2,
        anova_result.statistic
    ],
    "P_Value": [
        p1,
        p2,
        anova_result.pvalue
    ],
    "Significant_At_5_Percent": [
        p1 < 0.05,
        p2 < 0.05,
        anova_result.pvalue < 0.05
    ]
})

hypothesis_summary

In [ ]:
print("TOP 5 CRITICAL FINDINGS")
print("=" * 60)

print("\n1. Sales:")
print(
    f"Average sales = {df['Sales'].mean():.2f}, "
    f"median sales = {df['Sales'].median():.2f}."
)

print("\n2. Profit:")
print(
    f"Average profit = {df['Profit'].mean():.2f}, "
    f"median profit = {df['Profit'].median():.2f}."
)

print("\n3. Discount-Profit Relationship:")
print(
    f"Pearson correlation = {r1:.3f}, "
    f"p-value = {p1:.6f}."
)

print("\n4. Quantity-Sales Relationship:")
print(
    f"Pearson correlation = {r2:.3f}, "
    f"p-value = {p2:.6f}."
)

print("\n5. Category Profit:")
print(
    f"Highest average profit category: "
    f"{category_profit_avg.index[0]}"
)

## Top 5 Critical Business Findings

### 1. Sales Distribution
The analysis of sales values reveals the overall concentration and spread
of transaction values, including the presence of potentially high-value
transactions.

### 2. Profitability Pattern
Profit values show variation across individual transactions, highlighting
differences in profitability within the retail dataset.

### 3. Discount and Profit
The statistical correlation test evaluates whether discount levels are
associated with changes in profit.

### 4. Quantity and Sales
The relationship between quantity purchased and sales value provides
insight into how transaction volume relates to revenue.

### 5. Category-level Profitability
Category-level analysis shows differences in average profit across product
categories, providing a basis for further category performance analysis.

## Conclusion

This exploratory data analysis examined the Retail Orders dataset using
descriptive statistics, distribution analysis, box plots, correlation
analysis, multivariate visualizations, and statistical hypothesis testing.

The analysis provided insights into sales, profit, discount, quantity,
category performance, and regional performance.

Pearson correlation was used to investigate relationships between numerical
variables, while one-way ANOVA was applied to examine differences in
average profit across product categories.

The findings can support further business analysis by identifying
important relationships, unusual observations, and areas requiring
deeper investigation.